# Fine-tuning BAT on data subsets 

### Import libraries and load in config file

In [2]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var

from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *
from icu_benchmarks.models.dl_models.bat import * 

from icu_benchmarks.models.train import load_model
from pathlib import Path

import torch
import random
import numpy as np

vars_dict = {
    "GROUP": "stay_id",
    "SEQUENCE": "time",
    "LABEL": "label",
    "DYNAMIC": ["alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun", "ca", "cai", "ck", "ckmb", "cl",
        "crea", "crp", "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact", "lymph", "map", "mch", "mchc", "mcv",
        "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos", "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine",
        "wbc"],
    "STATIC": ["age", "sex", "height", "weight"],
}

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin")

/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

#### Load in pre-trained model

In [42]:
# Pre-trained on pooled mimic + miiv
#model_path = Path("/work3/s185395/yaib_logs/mimic_miiv/LOS/SSL_BAT_tuned_mimic_miiv/2025-08-27T10-33-15/repetition_0/fold_0/model.ckpt")
# Pre-trained on pooled eicu + mimic
#model_path = Path("/work3/s185395/yaib_logs/eicu_mimic/LOS/SSL_BAT_tuned_eicu_mimic/2025-08-28T01-27-19/repetition_0/fold_0/model.ckpt")
# Pre-trained on pooled eicu + miiv
model_path = Path("/work3/s185395/yaib_logs/eicu_miiv/LOS/SSL_BAT_tuned_eicu_miiv/2025-08-27T23-57-28/repetition_0/fold_0/model.ckpt")

ckpt = torch.load(model_path, map_location="cpu")
hparams = ckpt.get("hyper_parameters", {})

# Instantiate the model class (init args can be anything required)
model = SSL_BAT(**hparams) 

# Load only encoder weights
encoder_state_dict = {k.replace("model.encoder_class.", ""): v
                      for k, v in ckpt["state_dict"].items()
                      if k.startswith("model.encoder_class.")}

model.model.encoder_class.load_state_dict(encoder_state_dict)

# Extract encoder from SSL_BAT
pretrained_encoder = model.model.encoder_class

# Create classification model using the pretrained encoder
classification_model = EncoderPrediction(
    encoder_class=pretrained_encoder,
    prediction_head=BinaryClassificationHead,
    prediction_head_kwargs={"num_classes": 2}
)

use static


/tmp/ipykernel_3145671/4203960768.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location="cpu")


#### Load in saved out preprocessed data subsets

In [43]:
import polars as pl
from pathlib import Path

dataset = 'mimic' # eicu, miiv, mimic
size = 9506 # 100, 500, 1000, 2000, 3000, 5000, 7000, 9000. 9506
seed = 42 # 42, 84, 126, 168, 210 
subset_path = f"/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/{dataset}/{size}_{seed}" # !Change dataset subset here! 

# Set the directory where your Parquet files are saved
dir = Path(subset_path)

# Create the data dictionary in the format returned by preprocess_data()
data = {}

for split in ["train", "val", "test"]:
    outcome_path = dir / f"{split}_OUTCOME.parquet"
    features_path = dir / f"{split}_FEATURES.parquet"

    if outcome_path.exists() and features_path.exists():
        data[split] = {
            "OUTCOME": pl.read_parquet(outcome_path),
            "FEATURES": pl.read_parquet(features_path),
        }
        print(f"✅ Loaded {split} data")
    else:
        print(f"⚠️ Missing files for split '{split}'")


✅ Loaded train data
✅ Loaded val data
✅ Loaded test data


#### Create train, val and test datasets

In [44]:
from copy import deepcopy
from tqdm import tqdm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

from torch.utils.data import random_split
from icu_benchmarks.data.loader import *
from torch.utils.data import DataLoader

finetune_train_set = BATPolarsDataset(data=data, split="train", ram_cache=False, runmode=RunMode.classification, vars=vars_dict)
finetune_val_set = BATPolarsDataset(data=data, split="val", ram_cache=False, runmode=RunMode.classification,vars=vars_dict)
finetune_test_set = BATPolarsDataset(data=data, split="test", ram_cache=False, runmode=RunMode.classification, vars=vars_dict)

#### Function for fine-tuning on data subsets

In [29]:
def run_experiment(bz, lr, model_path, fine_tune_head, num_epochs = 200):

    # Setting seed for reproducibility (Only want variability in the subset datasets)
    seed = 42

    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # If using CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    g = torch.Generator()
    g.manual_seed(seed)

    bz = bz
    lr = lr

    ckpt = torch.load(model_path, map_location="cpu")
    hparams = ckpt.get("hyper_parameters", {})

    # Reset model parameters 
    # Instantiate the model class (init args can be anything required)
    model = SSL_BAT(**hparams) 

    # Load only encoder weights
    encoder_state_dict = {k.replace("model.encoder_class.", ""): v
                        for k, v in ckpt["state_dict"].items()
                        if k.startswith("model.encoder_class.")}

    model.model.encoder_class.load_state_dict(encoder_state_dict)

    # Extract encoder from SSL_BAT
    pretrained_encoder = model.model.encoder_class

    # Create classification model using the pretrained encoder
    classification_model = EncoderPrediction(
        encoder_class=pretrained_encoder,
        prediction_head=BinaryClassificationHead,
        prediction_head_kwargs={"num_classes": 2}
    )

    finetune_train_loader = DataLoader(finetune_train_set, batch_size=bz, shuffle=True, generator=g, 
                                    collate_fn=finetune_train_set.collate_fn_pad_to_longest_in_batch())
    finetune_val_loader = DataLoader(finetune_val_set, batch_size=bz, shuffle=True, generator=g, 
                                    collate_fn=finetune_val_set.collate_fn_pad_to_longest_in_batch())
    eval_loader = DataLoader(finetune_test_set, batch_size=bz, shuffle=False,
                            collate_fn=finetune_test_set.collate_fn_pad_to_longest_in_batch())

    print(f'Finetuening training dataset length: {len(finetune_train_set)}')
    print(f'Finetuening validation dataset length: {len(finetune_val_set)}')
    print(f'Finetuening test dataset length: {len(finetune_test_set)}')

    # Automatically select device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🔧 Using device: {device}")

    if fine_tune_head:
        for param in classification_model.parameters():
            param.requires_grad = False
        for param in classification_model.head.parameters():
            param.requires_grad = True
    else:
        for param in classification_model.parameters():
            param.requires_grad = True

    classification_model.to(device)
    optimizer = torch.optim.Adam(classification_model.parameters(), lr=lr)

    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

    loss_fn = torch.nn.CrossEntropyLoss()

    train_losses, val_losses = [], []
    train_aurocs, val_aurocs = [], []
    train_auprcs, val_auprcs = [], []

    # Set early stopping parameters
    patience = 3
    best_val_auprc = 0
    epochs_without_improvement = 0
    best_model_state = None

    for epoch in range(num_epochs):
        #current_lr = optimizer.param_groups[0]['lr']
        #print(f"📉 Current LR after epoch {epoch+1}: {current_lr:.6f}")
        # ======== TRAINING ========
        classification_model.train()
        total_train_loss = 0
        all_train_labels = []
        all_train_probs = []

        loop = tqdm(finetune_train_loader, desc=f"🔧 Fine-tuning Epoch {epoch+1}/{num_epochs}")
        for batch in loop:
            x, mask, label, times, static, *_ = batch
            x = x.to(device).float()
            mask = mask.to(device).float()
            times = times.to(device).float()
            static = static.to(device).float()
            label = label.to(device).long()

            optimizer.zero_grad()
            logits = classification_model(x, static=static, time=times, sensor_mask=mask)
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            probs = F.softmax(logits, dim=1)[:, 1]  # Probabilities for class 1

            all_train_labels.extend(label.cpu().numpy())
            all_train_probs.extend(probs.detach().cpu().numpy())
            loop.set_postfix(loss=loss.item())

        avg_train_loss = total_train_loss / len(finetune_train_loader)
        train_losses.append(avg_train_loss)

        train_auroc = roc_auc_score(all_train_labels, all_train_probs)
        train_auprc = average_precision_score(all_train_labels, all_train_probs)
        train_aurocs.append(train_auroc)
        train_auprcs.append(train_auprc)

        print(f"✅ Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f} | AUROC: {train_auroc:.4f} | AUPRC: {train_auprc:.4f}")

        # ======== VALIDATION ========
        classification_model.eval()
        total_val_loss = 0
        all_val_labels = []
        all_val_probs = []

        with torch.no_grad():
            for batch in finetune_val_loader:
                x, mask, label, times, static, *_ = batch
                x = x.to(device).float()
                mask = mask.to(device).float()
                times = times.to(device).float()
                static = static.to(device).float()
                label = label.to(device).long()

                logits = classification_model(x, static=static, time=times, sensor_mask=mask)
                loss = loss_fn(logits, label)
                total_val_loss += loss.item()

                probs = F.softmax(logits, dim=1)[:, 1]
                all_val_labels.extend(label.cpu().numpy())
                all_val_probs.extend(probs.cpu().numpy())

        avg_val_loss = total_val_loss / len(finetune_val_loader)
        val_losses.append(avg_val_loss)

        val_auroc = roc_auc_score(all_val_labels, all_val_probs)
        val_auprc = average_precision_score(all_val_labels, all_val_probs)
        val_aurocs.append(val_auroc)
        val_auprcs.append(val_auprc)

        print(f"🧪 Validation — Loss: {avg_val_loss:.4f} | AUROC: {val_auroc:.4f} | AUPRC: {val_auprc:.4f}")

        # ======== EARLY STOPPING & BEST MODEL SAVE ========
        scheduler.step()  # update learning rate based on val AUPRC

        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_model_state = deepcopy(classification_model.state_dict())
            epochs_without_improvement = 0
            print(f"📌 New best AUPRC: {best_val_auprc:.4f} — model checkpoint saved")
        else:
            epochs_without_improvement += 1
            print(f"⏳ No AUPRC improvement for {epochs_without_improvement} epoch(s)")

        if epochs_without_improvement >= patience:
            print(f"🛑 Early stopping triggered after {patience} epochs without improvement.")
            break

    # Restore best model after training
    classification_model.load_state_dict(best_model_state)

    # ======== TESTING ========
    print("\n🚀 Starting evaluation on test set...")

    classification_model.eval()
    total_test_loss = 0
    all_test_labels = []
    all_test_probs = []

    with torch.no_grad():
        test_loop = tqdm(eval_loader, desc="🧪 Evaluating on Test Set")
        for batch in test_loop:
            x, mask, label, times, static, *_ = batch
            x = x.to(device).float()
            mask = mask.to(device).float()
            times = times.to(device).float()
            static = static.to(device).float()
            label = label.to(device).long()

            logits = classification_model(x, static=static, time=times, sensor_mask=mask)
            loss = loss_fn(logits, label)
            total_test_loss += loss.item()

            probs = F.softmax(logits, dim=1)[:, 1]
            all_test_labels.extend(label.cpu().numpy())
            all_test_probs.extend(probs.cpu().numpy())

            test_loop.set_postfix(loss=loss.item())

    avg_test_loss = total_test_loss / len(eval_loader)
    test_auroc = roc_auc_score(all_test_labels, all_test_probs)
    test_auprc = average_precision_score(all_test_labels, all_test_probs)

    print(f"\n🎯 Test Set Results:")
    print(f"   Loss : {avg_test_loss:.4f}")
    print(f"   AUROC: {test_auroc:.4f}")
    print(f"   AUPRC: {test_auprc:.4f}")

    return {'bz': bz, 'lr': lr,'avg_test_loss': avg_test_loss, 'test_auroc': test_auroc, 'test_auprc': test_auprc}

### Grid hyperparameter tuning

In [45]:
# Full model or only head tuning 
fine_tune_head = True
#fine_tune_head = False

In [ ]:

# Learning rates and batch sizes to test
lrs = [1e-4, 5e-4, 1e-3, 5e-3, 1e-4, 5e-4]
batch_sizes = [24, 64]

# Store results
results = []

# Run the experiment for each (bz, lr) pair
for bz in batch_sizes:
    for lr in lrs:
        print(f"\n🚀 Running experiment with batch_size={bz}, learning_rate={lr}")
        result = run_experiment(bz, lr, model_path, fine_tune_head, num_epochs = 200) 
        results.append(result)


/tmp/ipykernel_3145671/1288834667.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location="cpu")



🚀 Running experiment with batch_size=64, learning_rate=0.0001
use static
Finetuening training dataset length: 9506
Finetuening validation dataset length: 2377
Finetuening test dataset length: 2971
🔧 Using device: cuda


🔧 Fine-tuning Epoch 1/200: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [00:13<00:00, 11.24it/s, loss=0.42]


✅ Epoch 1 - Train Loss: 0.5383 | AUROC: 0.3945 | AUPRC: 0.0956
🧪 Validation — Loss: 0.4701 | AUROC: 0.3054 | AUPRC: 0.0830
📌 New best AUPRC: 0.0830 — model checkpoint saved


🔧 Fine-tuning Epoch 2/200: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [00:13<00:00, 11.22it/s, loss=0.495]


...

✅ Epoch 14 - Train Loss: 0.3398 | AUROC: 0.6595 | AUPRC: 0.2678
🧪 Validation — Loss: 0.3423 | AUROC: 0.6563 | AUPRC: 0.2611
📌 New best AUPRC: 0.2611 — model checkpoint saved


🔧 Fine-tuning Epoch 15/200: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [00:13<00:00, 11.20it/s, loss=0.294]


✅ Epoch 15 - Train Loss: 0.3382 | AUROC: 0.6655 | AUPRC: 0.2724
🧪 Validation — Loss: 0.3409 | AUROC: 0.6609 | AUPRC: 0.2669
📌 New best AUPRC: 0.2669 — model checkpoint saved


🔧 Fine-tuning Epoch 16/200:  73%|█████████████████████████████████████████████████████████████████████████████▌                            | 109/149 [00:09<00:03, 11.33it/s, loss=0.224]


KeyboardInterrupt: 

### Teest specific combinations of batch size and lr 

In [41]:
top_configs = [
    #{'bz': 64, 'lr': 6e-3},
    {'bz': 24, 'lr': 1.5e-2},
    {'bz': 24, 'lr': 2e-2},
]

results = []

for config in top_configs:
    print(f"\n🚀 Re-running experiment: bz={config['bz']}, lr={config['lr']}")
    result = run_experiment(config['bz'], config['lr'], model_path, fine_tune_head)
    results.append(result)


/tmp/ipykernel_3145671/1288834667.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location="cpu")



🚀 Re-running experiment: bz=24, lr=0.015
use static
Finetuening training dataset length: 9506
Finetuening validation dataset length: 18141
Finetuening test dataset length: 22677
🔧 Using device: cuda


🔧 Fine-tuning Epoch 1/200: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 397/397 [00:09<00:00, 41.59it/s, loss=0.0343]


✅ Epoch 1 - Train Loss: 0.2024 | AUROC: 0.6991 | AUPRC: 0.1990
🧪 Validation — Loss: 0.1875 | AUROC: 0.7729 | AUPRC: 0.2549
📌 New best AUPRC: 0.2549 — model checkpoint saved


🔧 Fine-tuning Epoch 2/200: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 397/397 [00:09<00:00, 42.06it/s, loss=0.0893]


...

✅ Epoch 5 - Train Loss: 0.1825 | AUROC: 0.7715 | AUPRC: 0.2935
🧪 Validation — Loss: 0.2013 | AUROC: 0.7702 | AUPRC: 0.2545
⏳ No AUPRC improvement for 2 epoch(s)


🔧 Fine-tuning Epoch 6/200: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 397/397 [00:09<00:00, 41.54it/s, loss=0.0615]


✅ Epoch 6 - Train Loss: 0.1819 | AUROC: 0.7777 | AUPRC: 0.2879
🧪 Validation — Loss: 0.1892 | AUROC: 0.7794 | AUPRC: 0.2611
⏳ No AUPRC improvement for 3 epoch(s)
🛑 Early stopping triggered after 3 epochs without improvement.

🚀 Starting evaluation on test set...


🧪 Evaluating on Test Set: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 945/945 [00:21<00:00, 43.89it/s, loss=0.176]



🎯 Test Set Results:
   Loss : 0.1789
   AUROC: 0.7867
   AUPRC: 0.2986


In [38]:
results

[{'bz': 24,
  'lr': 0.007,
  'avg_test_loss': 0.17504883436338295,
  'test_auroc': 0.7923418755894192,
  'test_auprc': 0.3077918007645565},
 {'bz': 24,
  'lr': 0.009,
  'avg_test_loss': 0.17489515185040772,
  'test_auroc': 0.7949273177369615,
  'test_auprc': 0.30939019979098203},
 {'bz': 24,
  'lr': 0.005,
  'avg_test_loss': 0.17592111789478512,
  'test_auroc': 0.7937618727943063,
  'test_auprc': 0.30369093454283413}]

In [40]:
results

[{'bz': 24,
  'lr': 0.01,
  'avg_test_loss': 0.17520980472287173,
  'test_auroc': 0.7953020389625012,
  'test_auprc': 0.3093574170844281},
 {'bz': 24,
  'lr': 0.03,
  'avg_test_loss': 0.18817748307846682,
  'test_auroc': 0.778701858824711,
  'test_auprc': 0.29036684582913447}]